# 03: 主动学习闭环
## Uncertainty Sampling + DoE 选样

核心科学问题: **用 MB-PLS 的残差空间指导选择下一个双敲除**

流程:
1. MB-PLS 训练 → 残差空间
2. Uncertainty Sampling 选候选
3. DoE 设计矩阵优化
4. 输出推荐给合作者的 Top 10

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from chemocalib.models.mbpls import MultiBlockPLS, generate_toy_multiblock_data
from chemocalib.active_learning.uncertainty import UncertaintySampler
from chemocalib.active_learning.doe import ExperimentDesigner

sns.set_style('whitegrid')

In [ ]:
# 1. 训练 MB-PLS
blocks, y, _ = generate_toy_multiblock_data(n_samples=100, seed=42)
mbpls = MultiBlockPLS(n_components=5, block_names=['代谢组', '转录组', '蛋白组'])
mbpls.fit(blocks, y)

# 残差空间
residuals = mbpls.residual_space(blocks)

In [ ]:
# 2. 对比四种采样策略
strategies = ['residual', 'entropy', 'diversity', 'hybrid']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, (ax, strategy) in enumerate(zip(axes, strategies)):
    sampler = UncertaintySampler(strategy=strategy)
    u = sampler.compute_uncertainty(residuals)
    ax.bar(range(len(u)), np.sort(u)[::-1], color=plt.cm.Set2(i), width=0.8)
    ax.set_title(f'{strategy}')
    ax.set_xlabel('Rank')
    if i == 0:
        ax.set_ylabel('Uncertainty')

plt.suptitle('Uncertainty Scoring: 策略对比', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3. 构建候选基因对 + 主动选样
n_genes = 60
gene_pairs = [(f'G{i}', f'G{j}') for i in range(n_genes) for j in range(i+1, min(i+5, n_genes))]

sampler = UncertaintySampler(strategy='hybrid')
candidates = sampler.select_double_knockout_candidates(
    all_gene_pairs=gene_pairs,
    pair_features=np.random.randn(len(gene_pairs), 3),
    residuals=residuals,
    n_select=10,
    n_pool=200,
)

print('=' * 50)
print(' 主动学习推荐: Top 10 双敲除候选')
print('=' * 50)
print(candidates.to_string(index=False))

In [ ]:
# 4. DoE 实验设计矩阵
doe = ExperimentDesigner(n_factors=5, design_type='ccd')
design = doe.generate_design(n_center=3, alpha=1.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# CCD 设计空间
ax = axes[0]
ax.scatter(design[:, 0], design[:, 1], c='steelblue', s=80, edgecolor='k')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Factor 1')
ax.set_ylabel('Factor 2')
ax.set_title(f'CCD Design ({len(design)} runs)')

# 拉丁超立方
doe_lhs = ExperimentDesigner(n_factors=2)
lhs = doe_lhs.latin_hypercube(n_samples=30)
ax = axes[1]
ax.scatter(lhs[:, 0], lhs[:, 1], c='coral', s=60, edgecolor='k')
ax.set_xlabel('Factor 1')
ax.set_ylabel('Factor 2')
ax.set_title('Latin Hypercube (30 samples)')

plt.tight_layout()
plt.show()

In [ ]:
# 5. 闭环总结
print('\n' + '=' * 60)
print(' 闭环总结: 感知 → 预测 → 选样')
print('=' * 60)
print(f'''
  1. MB-PLS 建模: 3 块数据, {mbpls.n_components} 个潜变量
  2. 残差分析: uncertainty 评分
  3. 主动选样: 从 {len(gene_pairs)} 对中选出 Top {10}
  4. DoE 设计: {doe.design_type} 优化下一轮实验
  5. 给合作者: 做这 10 组实验即可!
''')